In [19]:
import pandas as pd 
import numpy as np
import joblib
import time
import sklearn.metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

## Load dataset generated by CovaS

In [20]:
def calculate_macro_tpr_fpr(voting_cm):
    num_classes = voting_cm.shape[0]
    tpr_list = []
    fpr_list = []

    for i in range(num_classes):
        TP = voting_cm[i, i]
        FN = np.sum(voting_cm[i, :]) - TP
        FP = np.sum(voting_cm[:, i]) - TP
        TN = np.sum(voting_cm) - (TP + FN + FP)

        TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
        FPR = FP / (FP + TN) if (FP + TN) > 0 else 0

        tpr_list.append(TPR)
        fpr_list.append(FPR)

    macro_tpr = np.mean(tpr_list)
    macro_fpr = np.mean(fpr_list)

    return macro_tpr, macro_fpr

train = pd.read_csv('/home/soc/hungnt/Modbus2023_600/gen_data/modbus_train_merged_2800_TabDiff_plus1400_label2.csv')
test = pd.read_csv('/dis/DS/hungnt/CICModbus2023/test_shap_58_600.csv')

In [22]:
X_train = train.drop(['Label'], axis=1)
y_train = train['Label']
X_test = test.drop(['Label'], axis=1)
y_test = test['Label']

In [4]:
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

## XGBoost

In [16]:
xgb_params = {
    'tree_method': 'gpu_hist',
    'predictor': 'gpu_predictor',
    'max_depth': 15,
    'n_estimators': 5000,
    'learning_rate': 0.25,
    'eval_metric': ['mlogloss', 'merror'],
    'objective': 'multi:softprob',
    'num_class': len(y_train.unique()),
    'booster': 'gbtree',
    'random_state': 42,
    'early_stopping_rounds': 50
}

train_weight = compute_sample_weight("balanced", y_train_split)
val_weight   = compute_sample_weight("balanced", y_val_split)

print("XGBClassifier Starting")
xgb_model = XGBClassifier(**xgb_params)

xgb_model.fit(
    X_train_split, y_train_split,
    sample_weight=train_weight,
    eval_set=[(X_val_split, y_val_split)],
    sample_weight_eval_set=[val_weight],
    verbose=False
)
print("XGBClassifier Finished")

xgb_start_time = time.time()
xgb_prediction = xgb_model.predict(X_test)
xgb_end_time = time.time()
xgb_time = xgb_end_time - xgb_start_time
xgb_acc = sklearn.metrics.accuracy_score(y_test, xgb_prediction)
xgb_precision = sklearn.metrics.precision_score(y_test, xgb_prediction, average='macro')
xgb_f1 = sklearn.metrics.f1_score(y_test, xgb_prediction, average='macro')
xgb_recall = sklearn.metrics.recall_score(y_test, xgb_prediction, average='macro')
xgb_cm = sklearn.metrics.confusion_matrix(y_test, xgb_prediction)

print("XGBoost report:")
print("XGBoost Time:", xgb_time)
print("XGBoost Accuracy:", xgb_acc)
print("XGBoost Precision:", xgb_precision)
print("XGBoost F1:", xgb_f1)
print("XGBoost Recall:", xgb_recall)
print("XGBoost CM:\n", xgb_cm)
xgb_tpr, xgb_fpr = calculate_macro_tpr_fpr(xgb_cm)
print(f'XGBoost Macro-average TPR: {xgb_tpr}')
print(f'XGBoost Macro-average FPR: {xgb_fpr}')
print(classification_report(y_test, xgb_prediction, digits=4))

XGBClassifier Starting
XGBClassifier Finished
XGBoost report:
XGBoost Time: 0.08144044876098633
XGBoost Accuracy: 0.954384186517993
XGBoost Precision: 0.9580372695799093
XGBoost F1: 0.9581564641635345
XGBoost Recall: 0.9585504347394368
XGBoost CM:
 [[594   4   0   0   0   0   2   0   0]
 [  1 514   5   0   0  10   0   0  21]
 [  0   9 553   0   0   4   0   0  34]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 455   0   1   0   0]
 [  1  14   0   0   0 396   4   0   6]
 [  2  10   1   0   0   8 426   0   0]
 [  2   0   0   0   0   0   0 451   0]
 [  1  13  20   0   0   7   0   0 351]]
XGBoost Macro-average TPR: 0.9585504347394368
XGBoost Macro-average FPR: 0.005775704889631016
              precision    recall  f1-score   support

           0     0.9884    0.9900    0.9892       600
           1     0.9113    0.9328    0.9220       551
           2     0.9551    0.9217    0.9381       600
           3     1.0000    1.0000    1.0000        26
           4     1.0000    0.9978 

In [17]:
joblib.dump(xgb_model, './models/xgb_TabDiff.pkl')

['./models/xgb_TabDiff.pkl']

## ExtraTree

In [18]:
et_params = {
    "n_estimators": 50,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy",
    "class_weight": "balanced"
}

print("ExtraTreesClassifier Starting")
et_model = ExtraTreesClassifier(**et_params)
et_model.fit(X=X_train, y=y_train)
et_start_time = time.time()
et_prediction = et_model.predict(X_test)
et_end_time = time.time()
et_time = et_end_time - et_start_time
print("ExtraTreesClassifier Finished")

et_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=et_prediction)
et_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=et_prediction)
et_fp = et_cm[0, 1]
print("ExtraTrees report:")
print("ExtraTrees Time:", et_end_time - et_start_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees CM:\n", et_cm)
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')
print(classification_report(y_test, et_prediction, digits=4))

ExtraTreesClassifier Starting


ExtraTreesClassifier Finished
ExtraTrees report:
ExtraTrees Time: 0.0592803955078125
ExtraTrees Accuracy: 0.9627470856563609
ExtraTrees Precision: 0.9659735269057066
ExtraTrees F1: 0.9662158125375065
ExtraTrees Recall: 0.9669822343102119
ExtraTrees CM:
 [[592   4   1   0   0   2   0   1   0]
 [  1 535   2   0   0   3   0   0  10]
 [  0  10 545   0   0  14   0   0  31]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 455   0   1   0   0]
 [  6  11   0   0   0 402   1   0   1]
 [  5   6   0   0   0   7 428   0   1]
 [  2   0   0   0   0   0   0 451   0]
 [  0   7  13   0   0   6   1   0 365]]
ExtraTrees Macro-average TPR: 0.9669822343102119
ExtraTrees Macro-average FPR: 0.004719954716551319
              precision    recall  f1-score   support

           0     0.9769    0.9867    0.9818       600
           1     0.9337    0.9710    0.9520       551
           2     0.9715    0.9083    0.9388       600
           3     1.0000    1.0000    1.0000        26
           4     1.0000

In [19]:
joblib.dump(et_model, './models/et_TabDiff.pkl')

['./models/et_TabDiff.pkl']

In [10]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn import metrics as sklearn

# === Hàm phụ (nếu đã có thì bỏ qua) ===
def calculate_macro_tpr_fpr(cm):
    C = cm.shape[0]
    tprs, fprs = [], []
    for c in range(C):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        tprs.append(tpr)
        fprs.append(fpr)
    return float(np.mean(tprs)), float(np.mean(fprs))

# === Tham số cố định ===
et_params = {
    "n_estimators": 70,        # sẽ thay trong vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy",
    "class_weight": "balanced"
}

START, END, STEP = 40, 60, 1  # có thể đổi STEP=1 để quét mịn

print("ExtraTreesClassifier Sweep Starting")

records = []
best_key = None
best_model, best_cm, best_pred = None, None, None
best_n, best_time = None, None

for n in range(START, END + 1, STEP):
    params = et_params.copy()
    params["n_estimators"] = n

    model = ExtraTreesClassifier(**params)
    model.fit(X_train, y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_test, pred)
    precision = sklearn.precision_score(y_test, pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_test, pred, average='macro')
    recall = sklearn.recall_score(y_test, pred, average='macro')
    cm = sklearn.confusion_matrix(y_test, pred)
    et_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": et_fp
    })

    key = (acc, f1, recall, -n)  # tie-break
    if best_key is None or key > best_key:
        best_key = key
        best_model, best_cm, best_pred = model, cm, pred
        best_n, best_time = n, t1 - t0

print("ExtraTreesClassifier Sweep Finished")

# === Tổng hợp kết quả ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], 
    ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === Report tốt nhất ===
et_acc = sklearn.accuracy_score(y_test, best_pred)
et_precision = sklearn.precision_score(y_test, best_pred, average='macro', zero_division=0)
et_f1 = sklearn.f1_score(y_test, best_pred, average='macro')
et_recall = sklearn.recall_score(y_test, best_pred, average='macro')
et_cm = best_cm
et_fp = et_cm[0, 1] if et_cm.shape == (2, 2) else None
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)

print("\n===== Best ExtraTrees report =====")
print(f"Best n_estimators: {best_n}")
print("ExtraTrees Time:", best_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees FP:", et_fp)
print("ExtraTrees CM:\n", et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')

# df_results.to_csv("./et_sweep_n_estimators_30_500.csv", index=False)


ExtraTreesClassifier Sweep Starting
ExtraTreesClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0            50  0.962747  0.966216      0.966982  0.057581
1            48  0.962240  0.965785      0.966595  0.055868
2            47  0.961733  0.965381      0.966228  0.054237
3            43  0.961480  0.965229      0.966188  0.053302
4            49  0.961480  0.965148      0.965960  0.056010
5            51  0.961480  0.965126      0.966023  0.053587
6            53  0.961480  0.965104      0.966023  0.055284
7            52  0.961480  0.965077      0.965878  0.056086
8            56  0.961480  0.965054      0.965988  0.066364
9            44  0.961227  0.964999      0.965923  0.054173

===== Best ExtraTrees report =====
Best n_estimators: 50
ExtraTrees Time: 0.057581424713134766
ExtraTrees Accuracy: 0.9627470856563609
ExtraTrees Precision: 0.9659735269057066
ExtraTrees F1: 0.9662158125375065
ExtraTrees Recall:

## RandomForest

In [23]:
rf_params = {
    "n_estimators": 850,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy",
    "class_weight": "balanced"
}

print("RandomForestClassifier Starting")
rf_model = RandomForestClassifier(**rf_params)
rf_model.fit(X=X_train, y=y_train)
rf_start_time = time.time()
rf_prediction = rf_model.predict(X_test)
rf_end_time = time.time()
rf_time = rf_end_time - rf_start_time
print("RandomForestClassifier Finished")

rf_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=rf_prediction)
rf_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=rf_prediction)
rf_fp = rf_cm[0, 1]
print("RandomForest report:")
print("RandomForest Time:", rf_end_time - rf_start_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest CM: \n", rf_cm)
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')
print(classification_report(y_test, rf_prediction, digits=4))

RandomForestClassifier Starting
RandomForestClassifier Finished
RandomForest report:
RandomForest Time: 0.37291574478149414
RandomForest Accuracy: 0.9645210339584389
RandomForest Precision: 0.9679489438298255
RandomForest F1: 0.9679414192035468
RandomForest Recall: 0.9685638922739996
RandomForest CM: 
 [[593   3   1   0   0   1   1   1   0]
 [  0 536   2   0   0   4   0   0   9]
 [  0  10 546   0   0   7   0   0  37]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 455   0   1   0   0]
 [  2  14   0   0   0 403   0   0   2]
 [  2  10   0   0   0   3 431   0   1]
 [  2   0   0   0   0   0   0 451   0]
 [  0  12  12   0   0   3   0   0 365]]
RandomForest Macro-average TPR: 0.9685638922739996
RandomForest Macro-average FPR: 0.004495631552859495
              precision    recall  f1-score   support

           0     0.9900    0.9883    0.9892       600
           1     0.9162    0.9728    0.9437       551
           2     0.9733    0.9100    0.9406       600
           3     1.0000

In [21]:
joblib.dump(rf_model, './models/rf_TabDiff.pkl')

['./models/rf_TabDiff.pkl']

In [11]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics as sklearn

# === Hàm phụ (nếu bạn đã có thì bỏ qua) ===
def calculate_macro_tpr_fpr(cm):
    """
    cm: confusion_matrix (C x C)
    Trả về (macro_TPR, macro_FPR)
    """
    C = cm.shape[0]
    tprs, fprs = [], []
    for c in range(C):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        tprs.append(tpr)
        fprs.append(fpr)
    return float(np.mean(tprs)), float(np.mean(fprs))

# === Tham số cố định ===
rf_params = {
    "n_estimators": 60,        # sẽ bị thay mỗi vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy",
    "class_weight": "balanced"
}

START, END, STEP = 840, 860, 1   # có thể đổi STEP=1 nếu muốn quét mịn hơn

print("RandomForestClassifier Sweep Starting")

records = []
best_key = None  # (acc, f1, recall, -n_estimators) để có tie-break rõ ràng
best_model = None
best_cm = None
best_pred = None
best_n = None
best_time = None

for n in range(START, END + 1, STEP):
    params = rf_params.copy()
    params["n_estimators"] = n

    model = RandomForestClassifier(**params)
    model.fit(X=X_train, y=y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_true=y_test, y_pred=pred)
    precision = sklearn.precision_score(y_true=y_test, y_pred=pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_true=y_test, y_pred=pred, average='macro')
    recall = sklearn.recall_score(y_true=y_test, y_pred=pred, average='macro')
    cm = sklearn.confusion_matrix(y_true=y_test, y_pred=pred)
    # Lưu ý: rf_fp = cm[0, 1] chỉ có ý nghĩa khi bài toán nhị phân và label 0/1 nằm đúng trục
    rf_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": rf_fp
    })

    # Tie-break: ưu tiên acc -> f1 -> recall; nếu vẫn hòa, chọn n_estimators nhỏ hơn
    key = (acc, f1, recall, -n)
    if (best_key is None) or (key > best_key):
        best_key = key
        best_model = model
        best_cm = cm
        best_pred = pred
        best_n = n
        best_time = t1 - t0

print("RandomForestClassifier Sweep Finished")

# === Tổng hợp kết quả thành DataFrame và in Top-10 theo Accuracy ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === In report mô hình tốt nhất (giữ format bạn đang dùng) ===
rf_acc = sklearn.accuracy_score(y_true=y_test, y_pred=best_pred)
rf_precision = sklearn.precision_score(y_true=y_test, y_pred=best_pred, average='macro', zero_division=0)
rf_f1 = sklearn.f1_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_recall = sklearn.recall_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_cm = best_cm
rf_fp = rf_cm[0, 1] if rf_cm.shape == (2, 2) else None
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)

print("\n===== Best RandomForest report =====")
print(f"Best n_estimators: {best_n}")
print("RandomForest Time:", best_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest FP:", rf_fp)
print("RandomForest CM:", rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')

RandomForestClassifier Sweep Starting
RandomForestClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0           850  0.964521  0.967941      0.968564  0.454182
1           852  0.964521  0.967941      0.968564  0.401869
2           853  0.964521  0.967941      0.968564  0.398108
3           854  0.964521  0.967941      0.968564  0.384799
4           855  0.964521  0.967941      0.968564  0.376179
5           856  0.964521  0.967941      0.968564  0.458988
6           840  0.964268  0.967664      0.968280  0.469801
7           857  0.964268  0.967664      0.968280  0.406885
8           859  0.964268  0.967664      0.968280  0.386297
9           848  0.964014  0.967455      0.967997  0.419146

===== Best RandomForest report =====
Best n_estimators: 850
RandomForest Time: 0.45418214797973633
RandomForest Accuracy: 0.9645210339584389
RandomForest Precision: 0.9679489438298255
RandomForest F1: 0.9679414192035468
Rand

In [13]:
import pandas as pd 
import numpy as np
import joblib
import time
import sklearn.metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

# lightgbm

In [22]:
import time
import lightgbm as lgb

lgbm_params = {
    'boosting_type': 'gbdt',
    'objective': 'multiclass',
    'num_class': int(y_train_split.nunique()),
    'learning_rate': 0.15,
    'max_depth': 5,
    'n_estimators': 3000,
    'device_type': 'cpu',
    'metric': ['multi_logloss', 'multi_error'],
    'random_state': 42,
    'class_weight': 'balanced'
}

print("LGBMClassifier Starting")
lgbm_model = lgb.LGBMClassifier(**lgbm_params)

lgbm_model.fit(
    X_train_split, y_train_split,
    eval_set=[(X_val_split, y_val_split)],
    eval_metric=['multi_logloss', 'multi_error'],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    verbose=False
)

lgbm_start_time = time.time()
lgbm_prediction = lgbm_model.predict(X_test)
lgbm_end_time = time.time()
lgbm_time = lgbm_end_time - lgbm_start_time
print("LGBMClassifier Finished")

lgbm_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=lgbm_prediction)
lgbm_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=lgbm_prediction, average='macro')
lgbm_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=lgbm_prediction, average='macro')
lgbm_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=lgbm_prediction, average='macro')
lgbm_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=lgbm_prediction)

print("LightGBM report:")
print("LightGBM Time:", lgbm_time)
print("LightGBM Accuracy:", lgbm_acc)
print("LightGBM Precision:", lgbm_precision)
print("LightGBM F1:", lgbm_f1)
print("LightGBM Recall:", lgbm_recall)
print("LightGBM CM:\n", lgbm_cm)
lgbm_tpr, lgbm_fpr = calculate_macro_tpr_fpr(lgbm_cm)
print(f'LightGBM Macro-average TPR: {lgbm_tpr}')
print(f'LightGBM Macro-average FPR: {lgbm_fpr}')
print(classification_report(y_test, lgbm_prediction, digits=4))

LGBMClassifier Starting


/home/soc/miniconda3/envs/soc/lib/python3.8/site-packages/lightgbm/sklearn.py:736: UserWarning: 'verbose' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose' argument is deprecated and will be removed in a future release of LightGBM. "


LGBMClassifier Finished
LightGBM report:
LightGBM Time: 0.023452281951904297
LightGBM Accuracy: 0.9548910288900152
LightGBM Precision: 0.9587951932584391
LightGBM F1: 0.9587617391398994
LightGBM Recall: 0.9589043879342669
LightGBM CM:
 [[592   3   2   0   0   2   1   0   0]
 [  0 517   3   0   0  14   1   0  16]
 [  0   9 553   0   0   2   0   0  36]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 455   0   0   0   1]
 [  0  17   0   0   0 401   3   0   0]
 [  1  12   0   0   0   5 429   0   0]
 [  1   0   0   0   2   0   0 450   0]
 [  1  12  28   0   0   6   0   0 345]]
LightGBM Macro-average TPR: 0.9589043879342669
LightGBM Macro-average FPR: 0.005723564288539607
              precision    recall  f1-score   support

           0     0.9950    0.9867    0.9908       600
           1     0.9070    0.9383    0.9224       551
           2     0.9437    0.9217    0.9325       600
           3     1.0000    1.0000    1.0000        26
           4     0.9956    0.9978    0.9967  

In [23]:
joblib.dump(lgbm_model, './models/lgbm_TabDiff.pkl')

['./models/lgbm_TabDiff.pkl']

# catboost

In [24]:
import time
from catboost import CatBoostClassifier

cat_params = {
    'task_type': 'GPU',
    'iterations': 3000,
    'depth': 5,
    'learning_rate': 0.2,
    'loss_function': 'MultiClass',
    'eval_metric': 'MultiClass',
    'random_seed': 42,
    'od_type': 'Iter', 
    'od_wait': 50,
    'verbose': False,
    'auto_class_weights': 'Balanced'
}

print("CatBoostClassifier Starting")
cat_model = CatBoostClassifier(**cat_params)
cat_model.fit(
    X_train_split, y_train_split,
    eval_set=(X_val_split, y_val_split),
    use_best_model=True,
    verbose=False
)

cat_start_time = time.time()
cat_prediction = cat_model.predict(X_test).astype(int).ravel()
cat_end_time = time.time()
cat_time = cat_end_time - cat_start_time
print("CatBoostClassifier Finished")

cat_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=cat_prediction)
cat_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=cat_prediction, average='macro')
cat_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=cat_prediction, average='macro')
cat_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=cat_prediction, average='macro')
cat_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=cat_prediction)

print("CatBoost report:")
print("CatBoost Time:", cat_time)
print("CatBoost Accuracy:", cat_acc)
print("CatBoost Precision:", cat_precision)
print("CatBoost F1:", cat_f1)
print("CatBoost Recall:", cat_recall)
print("CatBoost CM:\n", cat_cm)
cat_tpr, cat_fpr = calculate_macro_tpr_fpr(cat_cm)
print(f'CatBoost Macro-average TPR: {cat_tpr}')
print(f'CatBoost Macro-average FPR: {cat_fpr}')
print(classification_report(y_test, cat_prediction, digits=4))

CatBoostClassifier Starting


CatBoostClassifier Finished
CatBoost report:
CatBoost Time: 0.033544063568115234
CatBoost Accuracy: 0.9450076026355804
CatBoost Precision: 0.9491924374099993
CatBoost F1: 0.9492192352255694
CatBoost Recall: 0.9493610019302662
CatBoost CM:
 [[593   2   0   0   1   1   3   0   0]
 [  1 508  10   0   0   9   2   0  21]
 [  0  10 548   0   0   6   0   0  36]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 453   0   0   0   3]
 [  1  14   0   0   0 393   7   0   6]
 [  0  11   0   0   0   9 427   0   0]
 [  1   2   0   0   2   0   0 448   0]
 [  1  18  29   0   0  11   0   0 333]]
CatBoost Macro-average TPR: 0.9493610019302662
CatBoost Macro-average FPR: 0.006968122500182274
              precision    recall  f1-score   support

           0     0.9933    0.9883    0.9908       600
           1     0.8991    0.9220    0.9104       551
           2     0.9336    0.9133    0.9233       600
           3     1.0000    1.0000    1.0000        26
           4     0.9934    0.9934    0.99

In [25]:
joblib.dump(cat_model, './models/cat_TabDiff.pkl')

['./models/cat_TabDiff.pkl']